# 11 - Dataset Label Diagnostics

Bu notebook model egitimi yapmaz. Notebook 10 tarafindan uretilen sabit 0.50 esik sonuclarini Drive/lokal ciktilardan okur, sonra final test label kalitesini JSON raporlarindan ya da raw dataset uzerinden yeniden hesaplanan label/sequence diagnostiklerinden tamamlar.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

def find_project_root(start):
    start = Path(start).resolve()
    candidates = [start, start.parent, Path('/content/repo')]
    for candidate in candidates:
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_PROJECT = Path('/content/drive/MyDrive/ANN-Project')
    ROOT = find_project_root(Path('/content/repo') if Path('/content/repo').exists() else Path.cwd())
    RESULTS_DIR = DRIVE_PROJECT / 'Threshold_Free_Results'
    RAW_DATA_CANDIDATES = [
        DRIVE_PROJECT / 'data' / 'raw' / 'market_data_10y_enriched.csv',
        ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv',
    ]
else:
    ROOT = find_project_root(Path.cwd())
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    DRIVE_PROJECT = None
    RESULTS_DIR = ROOT / 'threshold_free_results'
    RAW_DATA_CANDIDATES = [
        ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv',
    ]

RESULTS_CSV = RESULTS_DIR / 'threshold_free_results.csv'
METRICS_DIR = RESULTS_DIR / 'artifacts' / 'metrics'
REPORT_PATHS = sorted(METRICS_DIR.glob('*_report.json')) if METRICS_DIR.exists() else []
RAW_DATA_PATH = next((p for p in RAW_DATA_CANDIDATES if p.exists()), None)

print(f'IN_COLAB    : {IN_COLAB}')
print(f'ROOT        : {ROOT}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print('\nDosya durumu:')
print(f'  results csv : {RESULTS_CSV} -> {RESULTS_CSV.exists()}')
print(f'  metrics dir : {METRICS_DIR} -> {METRICS_DIR.exists()}')
print(f'  report json : {len(REPORT_PATHS)} adet')
print(f'  raw dataset : {RAW_DATA_PATH if RAW_DATA_PATH else "BULUNAMADI"}')

## 1 - Hazir sonuc tablosunu oku

`threshold_free_results.csv` varsa metrikler oradan okunur. Dosya yoksa asagidaki aday listesi sadece diagnostik hesaplamak icin kullanilir; model metrikleri bos kalir.

In [ ]:
CANDIDATE_SPECS = [
    {
        'experiment_name': 'tf_cnn1d_c128_k3_l2_d2',
        'model_name': 'cnn1d',
        'architecture_variant': 'c128_k3_l2_d2',
        'scaler_name': 'standard',
        'threshold_quantile': 0.40,
        'lookback': 42,
    },
    {
        'experiment_name': 'tf_gru_h64_l2_d2',
        'model_name': 'gru',
        'architecture_variant': 'h64_l2_d2',
        'scaler_name': 'robust',
        'threshold_quantile': 0.20,
        'lookback': 42,
    },
    {
        'experiment_name': 'tf_cnn1d_c64_k5_l2_d2',
        'model_name': 'cnn1d',
        'architecture_variant': 'c64_k5_l2_d2',
        'scaler_name': 'standard',
        'threshold_quantile': 0.40,
        'lookback': 42,
    },
    {
        'experiment_name': 'tf_transformer_encoder_d64_h4_l1_d2',
        'model_name': 'transformer_encoder',
        'architecture_variant': 'd64_h4_l1_d2',
        'scaler_name': 'robust',
        'threshold_quantile': 0.20,
        'lookback': 5,
    },
]

spec_df = pd.DataFrame(CANDIDATE_SPECS)
spec_df['variant_tag'] = spec_df.apply(
    lambda r: f"q{int(round(float(r['threshold_quantile']) * 100)):02d}_lb{int(r['lookback'])}",
    axis=1,
)

def normalize_results_columns(df):
    df = df.copy()
    rename_map = {
        'model': 'model_name',
        'variant': 'architecture_variant',
        'fixed_threshold': 'probability_threshold',
        'test_bal_acc': 'test_balanced_accuracy',
        'cv_bal_acc': 'cv_balanced_accuracy_mean',
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    return df

if RESULTS_CSV.exists():
    results_df = normalize_results_columns(pd.read_csv(RESULTS_CSV))
    print(f'Sonuc CSV okundu: {RESULTS_CSV}')
else:
    results_df = pd.DataFrame()
    print('[UYARI] threshold_free_results.csv bulunamadi; metrikler JSON rapordan veya bos olarak gelecek.')

if results_df.empty:
    base_df = spec_df.copy()
else:
    base_df = results_df.merge(
        spec_df,
        on=['model_name', 'architecture_variant'],
        how='outer',
        suffixes=('', '_spec'),
    )
    for col in ['experiment_name', 'scaler_name', 'threshold_quantile', 'lookback', 'variant_tag']:
        spec_col = f'{col}_spec'
        if spec_col in base_df.columns:
            if col not in base_df.columns:
                base_df[col] = np.nan
            base_df[col] = base_df[col].combine_first(base_df[spec_col])
            base_df = base_df.drop(columns=[spec_col])

base_df[['experiment_name', 'model_name', 'architecture_variant', 'scaler_name', 'variant_tag', 'threshold_quantile', 'lookback']].to_string(index=False)

## 2 - JSON raporlarindan diagnostik cek

Rapor varsa final test label diagnostikleri dogrudan rapordan alinir. Bu en guvenilir kaynaktir cunku Notebook 10'un kosu sirasindaki threshold ve split bilgilerini birebir tasir.

In [ ]:
def load_reports(report_paths):
    reports = {}
    for path in report_paths:
        try:
            with open(path, encoding='utf-8') as f:
                report = json.load(f)
        except Exception as exc:
            print(f'[UYARI] Rapor okunamadi: {path.name} ({type(exc).__name__}: {exc})')
            continue
        exp_name = report.get('experiment_name') or path.stem.replace('_report', '')
        reports[exp_name] = report
    return reports

REPORTS = load_reports(REPORT_PATHS)
print(f'Yuklenen rapor sayisi: {len(REPORTS)}')
for name in sorted(REPORTS):
    print(f'  - {name}')

def _count_from_records(records, label_value):
    for item in records or []:
        if item.get('label') == label_value:
            return int(item.get('count', 0))
    return 0

def diagnostics_from_report(report):
    final_test = report.get('final_test', {})
    diag = final_test.get('test_label_diagnostics', {})
    all_records = diag.get('all_endpoint_labels', [])
    kept_records = diag.get('kept_binary_labels', [])

    n_before = sum(int(item.get('count', 0)) for item in all_records)
    n_after = sum(int(item.get('count', 0)) for item in kept_records)
    n_neutral = _count_from_records(all_records, -1)
    n_bear = _count_from_records(kept_records, 0)
    n_bull = _count_from_records(kept_records, 1)

    out = {
        'diagnostic_source': 'report_json',
        'label_threshold': final_test.get('threshold'),
        'n_sequences_before_drop': n_before,
        'n_sequences_after_drop': n_after,
        'neutral_drop_rate': n_neutral / n_before if n_before else np.nan,
        'bull_ratio': n_bull / n_after if n_after else np.nan,
        'bear_ratio': n_bear / n_after if n_after else np.nan,
        'n_bull': n_bull,
        'n_bear': n_bear,
        'n_neutral': n_neutral,
    }

    metrics = final_test.get('metrics', {})
    metric_map = {
        'test_mcc': 'mcc',
        'test_f1': 'f1',
        'test_balanced_accuracy': 'balanced_accuracy',
        'test_accuracy': 'accuracy',
        'test_roc_auc': 'roc_auc',
        'test_pr_auc': 'pr_auc',
    }
    for out_col, metric_key in metric_map.items():
        out[out_col] = metrics.get(metric_key)

    best_cv = report.get('best_cv_selection', {})
    cv_summary = best_cv.get('cv_summary', {})
    out['cv_mcc_mean'] = cv_summary.get('mcc_mean')
    out['cv_mcc_std'] = cv_summary.get('mcc_std')
    out['cv_f1_mean'] = cv_summary.get('f1_mean')
    out['cv_balanced_accuracy_mean'] = cv_summary.get('balanced_accuracy_mean')
    return out

## 3 - Rapor yoksa raw dataset ile label diagnostigi hesapla

Bu bolum sadece label/sequence sayilarini hesaplar. Model egitimi, tahmin veya threshold optimizasyonu yapmaz.

In [ ]:
DEFAULT_CONFIG = {
    'date_col': 'Date',
    'close_col': 'Nasdaq_Close',
    'feature_cols': [
        'VIX_Term_Structure',
        'Yield_Curve',
        'SKEW_Index',
        'Risk_Appetite_Ratio',
        'Crude_Oil',
        'VWAP_Deviation',
        'Volume_Momentum',
    ],
    'final_test_ratio': 0.15,
    'horizon': 1,
    'bull_label': 1,
    'bear_label': 0,
    'neutral_label': -1,
}

def load_project_config():
    config_path = ROOT / 'configs' / 'base.yaml'
    if not config_path.exists():
        return DEFAULT_CONFIG.copy()
    try:
        import yaml
        with open(config_path, encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
        return {
            'date_col': cfg['data']['date_col'],
            'close_col': cfg['data']['close_col'],
            'feature_cols': cfg['data']['feature_cols'],
            'final_test_ratio': float(cfg['split']['final_test_ratio']),
            'horizon': int(cfg['labeling']['horizon']),
            'bull_label': int(cfg['labeling']['bull_label']),
            'bear_label': int(cfg['labeling']['bear_label']),
            'neutral_label': int(cfg['labeling']['neutral_label']),
        }
    except Exception as exc:
        print(f'[UYARI] Config okunamadi, default kullaniliyor: {type(exc).__name__}: {exc}')
        return DEFAULT_CONFIG.copy()

CFG = load_project_config()

def compute_forward_return(close_series, horizon=1):
    return close_series.shift(-horizon) / close_series - 1.0

def compute_quantile_threshold(train_returns, quantile):
    valid = train_returns.dropna().abs()
    if len(valid) == 0:
        raise ValueError('No valid train returns for threshold computation.')
    return float(valid.quantile(float(quantile)))

def make_labels(returns, threshold, bull_label=1, bear_label=0, neutral_label=-1):
    labels = pd.Series(neutral_label, index=returns.index, dtype='int64')
    labels.loc[returns > threshold] = bull_label
    labels.loc[returns < -threshold] = bear_label
    return labels

def split_dev_test(df, test_ratio):
    n_rows = len(df)
    test_size = max(1, int(round(n_rows * test_ratio)))
    test_size = min(test_size, n_rows - 1)
    split_idx = n_rows - test_size
    return df.iloc[:split_idx].copy().reset_index(drop=True), df.iloc[split_idx:].copy().reset_index(drop=True)

def make_labelable_endpoint_indices(index_range, horizon):
    stop = index_range.stop - horizon
    if stop <= index_range.start:
        return range(index_range.start, index_range.start)
    return range(index_range.start, stop)

def load_clean_market_data():
    if RAW_DATA_PATH is None:
        return None
    df = pd.read_csv(RAW_DATA_PATH, parse_dates=[CFG['date_col']])
    df = df.sort_values(CFG['date_col']).reset_index(drop=True)
    required = [CFG['date_col'], CFG['close_col']] + CFG['feature_cols']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Raw dataset missing columns: {missing}')
    return df[required].copy().ffill().dropna(axis=0).reset_index(drop=True)

MARKET_DF = load_clean_market_data()
if MARKET_DF is None:
    print('[UYARI] Raw dataset bulunamadi; JSON raporu olmayan adaylarda diagnostik bos kalir.')
else:
    print(f'Raw dataset hazir: {MARKET_DF.shape[0]} satir, {MARKET_DF.shape[1]} kolon')

def diagnostics_from_raw(threshold_quantile, lookback):
    if MARKET_DF is None:
        return None

    dev_df, test_df = split_dev_test(MARKET_DF, CFG['final_test_ratio'])
    full_df = pd.concat([dev_df, test_df], axis=0, ignore_index=True)
    dev_returns = compute_forward_return(dev_df[CFG['close_col']], horizon=CFG['horizon'])
    label_threshold = compute_quantile_threshold(dev_returns, threshold_quantile)

    full_returns = compute_forward_return(full_df[CFG['close_col']], horizon=CFG['horizon'])
    labels = make_labels(
        full_returns,
        label_threshold,
        bull_label=CFG['bull_label'],
        bear_label=CFG['bear_label'],
        neutral_label=CFG['neutral_label'],
    )

    test_range = range(len(dev_df), len(full_df))
    labelable = make_labelable_endpoint_indices(test_range, CFG['horizon'])
    valid_endpoints = [idx for idx in labelable if idx >= int(lookback) - 1]
    y_all = labels.iloc[valid_endpoints]
    y_kept = y_all[y_all != CFG['neutral_label']]

    n_before = int(len(y_all))
    n_after = int(len(y_kept))
    n_neutral = int((y_all == CFG['neutral_label']).sum())
    n_bull = int((y_kept == CFG['bull_label']).sum())
    n_bear = int((y_kept == CFG['bear_label']).sum())

    return {
        'diagnostic_source': 'raw_dataset_recomputed',
        'label_threshold': label_threshold,
        'n_sequences_before_drop': n_before,
        'n_sequences_after_drop': n_after,
        'neutral_drop_rate': n_neutral / n_before if n_before else np.nan,
        'bull_ratio': n_bull / n_after if n_after else np.nan,
        'bear_ratio': n_bear / n_after if n_after else np.nan,
        'n_bull': n_bull,
        'n_bear': n_bear,
        'n_neutral': n_neutral,
    }

## 4 - Metrikler + dataset diagnostikleri

In [ ]:
def fill_first(row, columns):
    for col in columns:
        if col in row.index and pd.notna(row[col]):
            return row[col]
    return np.nan

rows = []
for _, row in base_df.iterrows():
    exp_name = row.get('experiment_name')
    report = REPORTS.get(exp_name)

    diag = diagnostics_from_report(report) if report else None
    if diag is None:
        diag = diagnostics_from_raw(row.get('threshold_quantile'), row.get('lookback'))
    if diag is None:
        diag = {'diagnostic_source': 'missing'}

    output = {
        'experiment_name': exp_name,
        'model_name': row.get('model_name'),
        'architecture_variant': row.get('architecture_variant'),
        'scaler_name': row.get('scaler_name'),
        'variant_tag': row.get('variant_tag'),
        'threshold_quantile': row.get('threshold_quantile'),
        'lookback': int(row.get('lookback')) if pd.notna(row.get('lookback')) else np.nan,
        'probability_threshold': row.get('probability_threshold'),
        'cv_mcc_mean': fill_first(row, ['cv_mcc_mean', 'cv_mcc']),
        'cv_mcc_std': row.get('cv_mcc_std'),
        'cv_f1_mean': fill_first(row, ['cv_f1_mean', 'cv_f1']),
        'cv_balanced_accuracy_mean': fill_first(row, ['cv_balanced_accuracy_mean', 'cv_bal_acc']),
        'test_mcc': row.get('test_mcc'),
        'test_f1': row.get('test_f1'),
        'test_balanced_accuracy': row.get('test_balanced_accuracy'),
        'test_accuracy': row.get('test_accuracy'),
        'test_roc_auc': row.get('test_roc_auc'),
        'test_pr_auc': row.get('test_pr_auc'),
    }

    for key, value in diag.items():
        if key not in output or pd.isna(output.get(key)):
            output[key] = value

    # If metrics were absent from CSV, fill them from report_json diagnostics.
    for metric_col in ['test_mcc', 'test_f1', 'test_balanced_accuracy', 'test_accuracy', 'test_roc_auc', 'test_pr_auc', 'cv_mcc_mean', 'cv_mcc_std', 'cv_f1_mean', 'cv_balanced_accuracy_mean']:
        if metric_col in diag and pd.isna(output.get(metric_col)):
            output[metric_col] = diag[metric_col]

    rows.append(output)

diagnostics_df = pd.DataFrame(rows)

display_cols = [
    'experiment_name', 'model_name', 'scaler_name', 'architecture_variant', 'variant_tag',
    'cv_mcc_mean', 'cv_mcc_std', 'test_mcc', 'test_f1', 'test_balanced_accuracy', 'test_roc_auc', 'test_pr_auc',
    'threshold_quantile', 'lookback', 'label_threshold', 'n_sequences_before_drop', 'n_sequences_after_drop',
    'neutral_drop_rate', 'bull_ratio', 'bear_ratio', 'diagnostic_source',
]
display_cols = [col for col in display_cols if col in diagnostics_df.columns]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda v: f'{v:.4f}' if pd.notna(v) else 'N/A')

diagnostics_view = diagnostics_df[display_cols].sort_values('test_mcc', ascending=False, na_position='last').reset_index(drop=True)
diagnostics_view

## 5 - Kisa grafikler

In [ ]:
plot_df = diagnostics_df.copy()
plot_df['short_name'] = plot_df['model_name'].astype(str) + '\n' + plot_df['architecture_variant'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

plot_df.sort_values('test_mcc', ascending=False).plot(
    x='short_name', y='test_mcc', kind='bar', ax=axes[0], legend=False, color='steelblue'
)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Final test MCC')
axes[0].set_xlabel('')
axes[0].set_ylabel('MCC')
axes[0].tick_params(axis='x', rotation=25)

if 'neutral_drop_rate' in plot_df.columns:
    plot_df.sort_values('neutral_drop_rate', ascending=False).plot(
        x='short_name', y='neutral_drop_rate', kind='bar', ax=axes[1], legend=False, color='darkorange'
    )
    axes[1].set_title('Neutral drop rate - final test')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Drop rate')
    axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

## 6 - Otomatik yorum

In [ ]:
def fmt_num(value, digits=4):
    return 'N/A' if pd.isna(value) else f'{value:.{digits}f}'

valid = diagnostics_df.copy()

if valid.empty:
    print('Yorum uretmek icin tablo bos.')
else:
    print('=== Kisa yorum ===')

    if valid['cv_mcc_mean'].notna().any():
        best_cv = valid.loc[valid['cv_mcc_mean'].idxmax()]
        print(
            f"- En iyi CV MCC: {best_cv['model_name']} / {best_cv['architecture_variant']} "
            f"({fmt_num(best_cv['cv_mcc_mean'])}); test MCC={fmt_num(best_cv.get('test_mcc'))}."
        )

    if valid['test_mcc'].notna().any():
        best_test = valid.loc[valid['test_mcc'].idxmax()]
        print(
            f"- En iyi test MCC: {best_test['model_name']} / {best_test['architecture_variant']} "
            f"({fmt_num(best_test['test_mcc'])}); q={best_test['threshold_quantile']}, lb={best_test['lookback']}."
        )

    auc_col = 'test_pr_auc' if valid['test_pr_auc'].notna().any() else 'test_roc_auc'
    if auc_col in valid and valid[auc_col].notna().any():
        best_auc = valid.loc[valid[auc_col].idxmax()]
        print(
            f"- En iyi {auc_col}: {best_auc['model_name']} / {best_auc['architecture_variant']} "
            f"({fmt_num(best_auc[auc_col])})."
        )

    if {'threshold_quantile', 'neutral_drop_rate', 'bull_ratio', 'bear_ratio'}.issubset(valid.columns):
        label_summary = (
            valid.groupby('threshold_quantile')[['neutral_drop_rate', 'bull_ratio', 'bear_ratio']]
                 .mean(numeric_only=True)
                 .reset_index()
                 .sort_values('threshold_quantile')
        )
        print('\n=== Quantile bazli label ozeti ===')
        print(label_summary.to_string(index=False))

    if {'lookback', 'n_sequences_after_drop'}.issubset(valid.columns):
        seq_summary = (
            valid.groupby('lookback')[['n_sequences_before_drop', 'n_sequences_after_drop', 'neutral_drop_rate']]
                 .mean(numeric_only=True)
                 .reset_index()
                 .sort_values('lookback')
        )
        print('\n=== Lookback bazli sequence ozeti ===')
        print(seq_summary.to_string(index=False))

    if valid['cv_mcc_mean'].notna().any() and valid['test_mcc'].notna().any():
        gap_df = valid.assign(cv_test_gap=valid['cv_mcc_mean'] - valid['test_mcc'])
        largest_gap = gap_df.loc[gap_df['cv_test_gap'].idxmax()]
        print(
            f"\n- CV-test farki en yuksek aday: {largest_gap['model_name']} / {largest_gap['architecture_variant']} "
            f"(gap={fmt_num(largest_gap['cv_test_gap'])}). Bu adayda label dagilimi ve neutral drop orani ozellikle kontrol edilmeli."
        )

## 7 - Analiz CSV olarak kaydet

In [ ]:
ANALYSIS_DIR = RESULTS_DIR / 'diagnostics'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = ANALYSIS_DIR / 'dataset_label_diagnostics_summary.csv'
diagnostics_df.to_csv(OUT_CSV, index=False)
print(f'Kaydedildi: {OUT_CSV}')